# 5장 3강 : 클로저와 데코레이터

#### 1. 파이썬 함수의 일급 객체 특성과 고차 함수

#### 2. 클로저(Closure)
- 고차 함수(Higher Order Function)

In [10]:
def outer():
    num1 = 10

    def inner():
        num2 = 20
        print(num1 + num2)

    inner()

inner = outer() # outer는 inner를 호출만 하고 반환하지 않으므로 결과는 None

30


위 예제는 `inner()`를 `outer()` 안에서 바로 실행하므로 클로저를 반환하는 예제는 아니다. 클로저로 사용하려면 내부 함수가 바깥 함수의 지역 이름을 참조하고, 바깥 함수가 내부 함수 객체를 반환해야 한다. 바깥 함수의 프레임 전체가 그대로 남는다고 보기보다 내부 함수가 필요한 자유 변수를 클로저 셀로 참조한다고 이해하는 편이 정확하다.

In [ ]:
def make_adder(num1):
    def add_number(num2):
        return num1 + num2

    return add_number

add10 = make_adder(10)
print(add10(20))
print(add10.__closure__)
print(add10.__closure__[0].cell_contents)

In [2]:
def add(num1):

    def _add(num2):

        def __add(num3):
            return num1 + num2 + num3

        return __add
    
    return _add

result = add(10)(20)(30)
result

60

In [12]:
add10 = add(10)
add10(30)(40)

80

In [13]:
add10_20 = add(10)(20)
add10_20(30)

60

#### 3. 데코레이터(Decorator)
- 함수를 입력받아 공통 기능을 추가한 다른 호출 가능한 객체로 교체하는 구조
- 데코레이터와 재귀는 서로 다른 개념이다. 데코레이터를 적용한 함수가 자기 자신을 다시 호출할 수는 있지만, 데코레이터 자체가 재귀를 의미하지는 않는다.

In [16]:
def factorial1(num):
    result = 1

    for i in range(1, num + 1):
        result *= i

    return result

factorial1(10)

3628800

In [18]:
def factorial2(num):
    if num < 1:
        return 1
    
    return num * factorial2(num - 1)

factorial2(10)

3628800

In [41]:
import time

start = time.time()

result = factorial1(10)
print(result)

end = time.time()
print("걸린 시간 : ", end - start)

3628800
걸린 시간 :  0.00015044212341308594


In [ ]:
start = time.time() # 공통 기능

result = factorial2(10)  # 핵심 기능
print(result)

end = time.time() # 공통 기능
print("걸린 시간 : ", end - start)

3628800
걸린 시간 :  0.00023794174194335938


In [95]:
import time

def time_checker(callback, num):
    start = time.time()

    result = callback(num)

    end = time.time()
    print("걸린 시간 : ", end - start)

    return result

time_checker(factorial1, 10)

걸린 시간 :  1.1920928955078125e-06


3628800

In [106]:
import time

def time_check2(callback):

    def wrapper(*args, **kwargs):
        start = time.time()

        result = callback(*args, **kwargs)

        end = time.time()
        print("걸린 시간 : ", end - start)

        return result

    return wrapper

@time_check2
def factorial3(num):
    if num < 1:
        return 1
    
    return num * factorial3(num - 1)

factorial3(10)

걸린 시간 :  2.384185791015625e-07
걸린 시간 :  0.00010013580322265625
걸린 시간 :  0.00011205673217773438
걸린 시간 :  0.00012350082397460938
걸린 시간 :  0.00013399124145507812
걸린 시간 :  0.0001456737518310547
걸린 시간 :  0.00015544891357421875
걸린 시간 :  0.0001659393310546875
걸린 시간 :  0.0001761913299560547
걸린 시간 :  0.0001862049102783203
걸린 시간 :  0.0001971721649169922


3628800

In [107]:
@time_check2
def add(num1, num2):
    return num1 + num2

add(num1=10, num2=20)

걸린 시간 :  7.152557373046875e-07


30

In [104]:
a = [10, 20]

def minus(num1, num2):
    return num1 - num2

minus(*a)

-10

#### 4. `functools.wraps`로 원래 함수 정보 보존
데코레이터가 원래 함수를 `wrapper`로 교체하면 함수 이름과 문서 문자열도 `wrapper`의 정보로 보일 수 있다. `@wraps(callback)`을 적용하면 원래 함수의 메타데이터를 보존할 수 있다.

In [ ]:
from functools import wraps
from time import perf_counter

def time_checker_with_wraps(callback):
    @wraps(callback)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = callback(*args, **kwargs)
        end = perf_counter()
        print("걸린 시간 :", end - start)
        return result

    return wrapper

@time_checker_with_wraps
def multiply(num1, num2):
    """두 수를 곱한다."""
    return num1 * num2

print(multiply(10, 20))
print(multiply.__name__)
print(multiply.__doc__)